# Challenge Weighted Max-Cut QAOA on QCS

This notebook solves the challenge weighted Max-Cut objective on a user-selected region of the grid graph.

$$C(z)=\frac{1}{2}\sum_{(i,j)\in E} w_{ij}(1-z_i z_j),$$

It lets you choose any region by explicit graph node list and automatically switches to a light-cone preconditioned QAOA workflow when the selected region is too large for direct QAOA.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from pyquil import get_qc

from challenge_qaoa import load_problem_graph, plot_landscape, solve_selected_region

## Select a QPU / QVM target

Choose a QPU, then use either that QPU directly or the matching `-qvm` service.

In [2]:
import os
import sys

try:
    sys.path.append(os.path.abspath(os.path.join(os.environ["HOME"], "tutorials")))
    from qpus_and_qvms import select_available_qpu

    quantum_processor_id = select_available_qpu()
except Exception:
    quantum_processor_id = "Ankaa-3"

execution_target = "qpu"  # set to "qpu" to run on hardware
qc_name = f"{quantum_processor_id}-qvm" if execution_target == "qvm" else quantum_processor_id
qc = get_qc(qc_name)

print(f"Quantum processor: {quantum_processor_id}")
print(f"Execution target: {qc_name}")

Selecting available QPU: Ankaa-3
Quantum processor: Ankaa-3
Execution target: Ankaa-3


## Load the challenge graph

In [3]:
graph_path, graph = load_problem_graph()
print(f"Loaded graph from: {Path(graph_path).resolve()}")
print(f"Nodes: {graph.number_of_nodes()}  Edges: {graph.number_of_edges()}")

try:
    region_summary = pd.read_csv("problem_b_region_summary.csv")
    display(region_summary[["region_id", "region_category", "node_count", "frustration_score"]].head(12))
except FileNotFoundError:
    print("problem_b_region_summary.csv not found; continuing without the optional region summary preview.")

Loaded graph from: /home/jovyan/019c9083-9e69-72f0-b313-85026d9a88aa.parquet
Nodes: 180  Edges: 226


,region_id,region_category,node_count,frustration_score
0,R00,qaoa,8,0.207368
1,R01,classical,12,0.014996
2,R02,classical,8,0.038844
3,R03,qaoa,9,0.281378
4,R04,quantum preconditioning,24,0.294556
5,R05,quantum preconditioning,15,0.155988
6,R06,quantum preconditioning,11,0.162330
7,R07,quantum preconditioning,21,0.206468
8,R08,qaoa,7,0.198597
9,R09,qaoa,9,0.158122


## Choose a region by node list

Direct QAOA is used for regions with at most 10 nodes. Larger regions default to the light-cone preconditioned QAOA workflow.

In [4]:
selected_nodes = sorted(pd.read_csv("problem_b_region_node_map.csv").query("region_id == 'R04'")["node_id"].astype(int).tolist())
method = "preconditioned"  # "auto", "qaoa", or "preconditioned"

qaoa_width = 8
qaoa_shots = 250
precondition_width = 9
precondition_shots = 128
seed = 42

result = solve_selected_region(
    graph,
    selected_nodes,
    method=method,
    qc_name=qc_name,
    width=5,
    shots=qaoa_shots,
    seed=seed,
    precondition_width=precondition_width,
    precondition_shots=precondition_shots,
)

result.as_dict()

[preconditioned] Building and exporting preconditioned graph only (no classical solve).
[preconditioned] Preparing 86 light-cone QAOA evaluations from 96 candidate node pairs.


Preconditioning cones:   0%|          | 0/86 [00:00<?, ?it/s]

[preconditioned] Evaluating cone 1/86 (nodes=4, pairs=1).
[preconditioned] Evaluating cone 2/86 (nodes=4, pairs=1).
[preconditioned] Evaluating cone 3/86 (nodes=4, pairs=1).
[preconditioned] Evaluating cone 4/86 (nodes=4, pairs=1).
[preconditioned] Evaluating cone 5/86 (nodes=4, pairs=3).
[preconditioned] Evaluating cone 6/86 (nodes=5, pairs=1).
[preconditioned] Evaluating cone 7/86 (nodes=5, pairs=1).
[preconditioned] Evaluating cone 8/86 (nodes=5, pairs=3).
[preconditioned] Evaluating cone 9/86 (nodes=5, pairs=1).
[preconditioned] Evaluating cone 10/86 (nodes=5, pairs=1).
[preconditioned] Evaluating cone 11/86 (nodes=5, pairs=1).
[preconditioned] Evaluating cone 12/86 (nodes=5, pairs=1).
[preconditioned] Evaluating cone 13/86 (nodes=5, pairs=1).
[preconditioned] Evaluating cone 14/86 (nodes=5, pairs=1).
[preconditioned] Evaluating cone 15/86 (nodes=5, pairs=1).
[preconditioned] Evaluating cone 16/86 (nodes=5, pairs=1).
[preconditioned] Evaluating cone 17/86 (nodes=5, pairs=1).
[preco

response failed classification=Error: transport error latency=30024 ms


QpuApiError: Call failed during gRPC request: status: Cancelled, message: "Timeout expired", details: [], metadata: MetadataMap { headers: {} }

In [ ]:
print(f"Method used: {result.method}")
print(f"Cut value: {result.cut_value:.6f}")
print(f"Left partition (+1): {list(result.left_partition)}")
print(f"Right partition (-1): {list(result.right_partition)}")

assignment_frame = pd.DataFrame(
    {
        "node": list(result.selected_nodes),
        "spin": [result.assignment[node] for node in result.selected_nodes],
        "partition": ["left" if result.assignment[node] > 0 else "right" for node in result.selected_nodes],
    }
)
assignment_frame

In [ ]:
if result.landscape is not None:
    plot_landscape(
        result.landscape,
        beta_values=result.beta_values,
        gamma_values=result.gamma_values,
        title=(
            f"Challenge Weighted Max-Cut QAOA\\n"
            f"{qc_name}\\n"
            f"nodes = {list(result.selected_nodes)}"
        ),
    )
else:
    print("No direct QAOA landscape is produced for the preconditioned solve path.")

In [ ]:
output_path = Path("challenge_region04_result.csv")

solution_export_frame = assignment_frame.copy()
solution_export_frame["method"] = result.method
solution_export_frame["qc_name"] = result.qc_name
solution_export_frame["cut_value"] = result.cut_value
solution_export_frame["sample_cut_value"] = result.sample_cut_value
solution_export_frame["mean_objective"] = result.mean_objective
solution_export_frame["selected_nodes"] = ",".join(str(node) for node in result.selected_nodes)
solution_export_frame.to_csv(output_path, index=False)

print(f"Saved region solve result to {output_path.resolve()}")
solution_export_frame